# 模块 6：视觉 PID 纯算法示例

本 notebook 直接复用 `pid_controller.py`，展示中心目标、方向符号、远近目标、多目标选择和 0.2 s 超时零输出。不导入 ROS 消息。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'code' / 'pid_controller.py').is_file():
    root = root.parent
assert (root / 'code' / 'pid_controller.py').is_file(), '未找到项目根目录'
sys.path.insert(0, str(root / 'code'))

from datatypes import VisualTrack
from pid_controller import PIDController, command_from_track, on_timer, select_track

def controllers():
    return (PIDController(0.01, 0.0, 0.0, 1.0),
            PIDController(0.01, 0.0, 0.0, 1.0),
            PIDController(1.0, 0.0, 0.0, 2.0))

def command(track):
    return command_from_track(track, *controllers(), dt=0.02)

## 1. 中心目标与方向符号

右侧/下方目标产生负 `yaw_rate`/负 `velocity.z`，左侧/上方则相反。

In [ ]:
cases = [
    ('center', VisualTrack(1, (270.0, 190.0, 100.0, 100.0), 0.10)),
    ('right/down', VisualTrack(2, (430.0, 310.0, 100.0, 100.0), 0.10)),
    ('left/up', VisualTrack(3, (110.0, 70.0, 100.0, 100.0), 0.10)),
]
commands = {}
for name, target in cases:
    commands[name] = command(target)
    print('%-10s velocity=%s yaw_rate=%+.3f' % (name, commands[name].velocity, commands[name].yaw_rate))
assert commands['center'].velocity == (0.0, 0.0, 0.0)
assert commands['center'].yaw_rate == 0.0
assert commands['right/down'].yaw_rate < 0 and commands['right/down'].velocity_z < 0
assert commands['left/up'].yaw_rate > 0 and commands['left/up'].velocity_z > 0

## 2. 远近目标的深度速度

直接面积误差策略为 `desired_area - max(area_ratio, min_area)`：远目标前进，达到期望面积时为零，进入末端阈值时使用末端速度。

In [ ]:
for label, area in [('far', 0.01), ('desired', 0.10), ('near', 0.40)]:
    out = command(VisualTrack(10, (270.0, 190.0, 100.0, 100.0), area))
    print('%-8s area=%.2f velocity.x=%+.3f' % (label, area, out.velocity_x))
    if label == 'far':
        assert out.velocity_x > 0
    elif label == 'desired':
        assert out.velocity_x == 0
    else:
        assert out.velocity_x == 0

## 3. 多目标选择、短时保持与超时停止

选择面积最大者，同面积取最小 ID。空数组短时保持上次指令，超过 0.2 s 输出零速度。

In [ ]:
targets = [
    VisualTrack(7, (270.0, 190.0, 100.0, 100.0), 0.03),
    VisualTrack(4, (270.0, 190.0, 100.0, 100.0), 0.08),
    VisualTrack(2, (270.0, 190.0, 100.0, 100.0), 0.08),
]
selected = select_track(targets)
fresh = on_timer(targets, 0.0, None, controllers=controllers())
held = on_timer([], 0.10, fresh, controllers=controllers())
stopped = on_timer([], 0.21, fresh, controllers=controllers())
print('selected id:', selected.id)
print('fresh      :', fresh)
print('held       :', held)
print('stopped    :', stopped)
assert selected.id == 2 and held == fresh
assert stopped.velocity == (0.0, 0.0, 0.0) and stopped.yaw_rate == 0.0
assert stopped.coordinate_frame == 9 and stopped.type_mask == 1479